In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio
import kairo

pio.renderers.default = "notebook_connected"
pd.set_option("display.precision", 3)

LOG = "../../data/thesis/general/running_example/L3.xes"
META = ["case:concept:name", "time:timestamp"]
AMOUNT = pd.Timestamp("2020-09-30 18:00", tz="UTC")

In [2]:
log = kairo.read_log(LOG, resource="org:resource", event_duration="event:duration_min")
print(log["case:concept:name"].nunique(), "cases,", len(log), "events")

4920 cases, 24600 events


In [3]:
features = kairo.compute_features_inter(log, feature_window="12h",
                                        features=["active_cases", "new_arrivals", "mean_delta_t", "attr_means"],
                                        case_attributes={"case:amount": "numerical"})
print(len(features), "feature windows,", features.shape[1] - 1, "feature columns")

731 feature windows, 4 feature columns


In [4]:
windows = pd.to_datetime(["2020-09-29 12:00", "2020-09-30 00:00", "2020-10-01 00:00", "2020-10-01 12:00"]).tz_localize("UTC")
features[features["time:timestamp"].isin(windows)].set_index("time:timestamp").T

time:timestamp,2020-09-29 12:00:00+00:00,2020-09-30 00:00:00+00:00,2020-10-01 00:00:00+00:00,2020-10-01 12:00:00+00:00
active_cases,6.000,9.000,7.000,8.000
new_arrivals,5.000,7.000,4.000,7.000
mean_delta_t,50.656,45.220,44.289,42.503
attr_means:case:amount,912.458,931.223,2375.814,2599.657


In [5]:
values = features.drop(columns="time:timestamp")
unscaled = kairo.compute_pca(features)
print("standard deviations:", values.std().round(3).to_dict())
print("explained variance % without scaling:", np.round(100 * unscaled.explained_variance_ratio_, 2))
values.corr().round(3)

standard deviations: {'active_cases': 2.854, 'new_arrivals': 2.552, 'mean_delta_t': 3.123, 'attr_means:case:amount': 673.998}
explained variance % without scaling: [99.99  0.    0.    0.  ]


,active_cases,new_arrivals,mean_delta_t,attr_means:case:amount
active_cases,1.000,0.896,0.044,-0.006
new_arrivals,0.896,1.000,0.071,-0.009
mean_delta_t,0.044,0.071,1.000,0.071
attr_means:case:amount,-0.006,-0.009,0.071,1.000


In [6]:
minmax = kairo.standardize(features, method="minmax")
gaps = minmax["mean_delta_t"]
print(features[values["active_cases"] == 0].to_string())
print("5% and 95% quantiles of the min-max gaps:", gaps.quantile([0.05, 0.95]).round(3).to_list(),
      "share in [0.73, 0.89]:", gaps.between(0.73, 0.89).mean().round(3))
minmax.drop(columns="time:timestamp").std().round(3)

               time:timestamp  active_cases  new_arrivals  mean_delta_t  attr_means:case:amount
398 2020-07-18 00:00:00+00:00           0.0           0.0           0.0                     0.0
5% and 95% quantiles of the min-max gaps: [0.735, 0.883] share in [0.73, 0.89]: 0.917


active_cases              0.150
new_arrivals              0.170
mean_delta_t              0.055
attr_means:case:amount    0.226
dtype: float64

In [7]:
scaled = kairo.standardize(features, method="zscore")
print("smallest standardized mean gap", scaled["mean_delta_t"].min().round(2))
scaled[scaled["time:timestamp"].isin(windows)].set_index("time:timestamp").T

smallest standardized mean gap -14.53


time:timestamp,2020-09-29 12:00:00+00:00,2020-09-30 00:00:00+00:00,2020-10-01 00:00:00+00:00,2020-10-01 12:00:00+00:00
active_cases,-0.837,0.214,-0.487,-0.137
new_arrivals,-0.678,0.106,-1.070,0.106
mean_delta_t,1.689,-0.052,-0.350,-0.921
attr_means:case:amount,-0.694,-0.666,1.478,1.810


In [8]:
pca = kairo.compute_pca(scaled)
errors = kairo.compute_pca_reconstruction_errors(scaled, pca)
standardized = scaled.drop(columns="time:timestamp")
print("explained variance %:", np.round(100 * pca.explained_variance_ratio_, 1),
      "cumulative %:", np.round(100 * pca.explained_variance_ratio_.cumsum(), 1))
print("error without components", round(((standardized - standardized.mean()) ** 2).sum(axis=1).mean(), 3),
      "with components", errors.round(3).to_list())
pd.DataFrame(pca.components_.T, index=pca.feature_names_in_, columns=[f"pc_{i + 1}" for i in range(pca.n_components_)]).round(3)

explained variance %: [47.6 26.7 23.1  2.6] cumulative %: [ 47.6  74.3  97.4 100. ]
error without components 3.995 with components [2.094, 1.027, 0.103, 0.0]


,pc_1,pc_2,pc_3,pc_4
active_cases,0.704,-0.050,-0.062,-0.706
new_arrivals,0.705,-0.033,-0.034,0.708
mean_delta_t,0.089,0.687,0.721,-0.022
attr_means:case:amount,-0.005,0.724,-0.690,0.005


In [9]:
min_samples = 8
kairo.plot_dbscan_k_distance(scaled, min_samples=min_samples).show()

In [10]:
after = scaled["time:timestamp"] >= AMOUNT
candidates = {}

for eps in [0.6, 0.7, 0.8, 0.9, 1.0]:
    dbscan = kairo.compute_dbscan(scaled, eps=eps, min_samples=min_samples)
    states = kairo.get_dbscan_states(scaled, dbscan)
    clusters = states["cluster"].max() + 1
    candidates[eps] = {
        "clusters": clusters,
        "noise before": int(((states["cluster"] == -1) & ~after).sum()),
        "noise after": int(((states["cluster"] == -1) & after).sum()),
        "silhouette": kairo.compute_silhouette(scaled, states) if clusters > 1 else None,
    }
pd.DataFrame(candidates).T

,clusters,noise before,noise after,silhouette
0.6,3.0,32.0,72.0,0.333
0.7,2.0,23.0,35.0,0.382
0.8,2.0,14.0,12.0,0.373
0.9,2.0,6.0,6.0,0.365
1.0,1.0,5.0,5.0,NaN


In [11]:
eps = 0.7
dbscan = kairo.compute_dbscan(scaled, eps=eps, min_samples=min_samples)
states = pd.concat([scaled, kairo.get_dbscan_states(scaled, dbscan)], axis=1)

first_visit = states.loc[states["cluster"] != -1].drop_duplicates("cluster")["cluster"].tolist()
states["state"] = states["cluster"].map({cluster: number for number, cluster in enumerate(first_visit, start=1)}).fillna(0).astype(int)
pd.crosstab(np.where(after, "after", "before"), states["state"])

state,0,1,2
row_0,,,
after,35,0,148
before,23,525,0


In [12]:
values = states.drop(columns=["time:timestamp", "cluster", "state"])
largest = values.abs().max(axis=1)
print("median of the largest standardized feature, noise:", round(largest[states["state"] == 0].median(), 2),
      "clusters:", round(largest[states["state"] != 0].median(), 2))
states[states["time:timestamp"].between("2020-09-29", "2020-10-02")]

median of the largest standardized feature, noise: 2.12 clusters: 1.19


,time:timestamp,active_cases,new_arrivals,mean_delta_t,attr_means:case:amount,cluster,state
544,2020-09-29 00:00:00+00:00,1.265,0.497,0.244,-0.553,0,1
545,2020-09-29 12:00:00+00:00,-0.837,-0.678,1.689,-0.694,0,1
546,2020-09-30 00:00:00+00:00,0.214,0.106,-0.052,-0.666,0,1
547,2020-09-30 12:00:00+00:00,-0.487,-0.286,-0.580,0.526,-1,0
548,2020-10-01 00:00:00+00:00,-0.487,-1.070,-0.350,1.478,1,2
549,2020-10-01 12:00:00+00:00,-0.137,0.106,-0.921,1.810,1,2
550,2020-10-02 00:00:00+00:00,0.214,0.106,-0.853,1.544,1,2


In [13]:
cluster_states = states[["cluster"]]
print(pd.Series({
    "states without noise": states["cluster"].max() + 1,
    "silhouette": kairo.compute_silhouette(scaled, cluster_states),
    "Calinski-Harabasz": kairo.compute_calinski_harabasz(scaled, cluster_states),
}).round(3).to_string())
print(kairo.get_dbscan_state_distances(dbscan).round(2))

states without noise      2.000
silhouette                0.382
Calinski-Harabasz       271.435
      0     1
0  0.00  2.25
1  2.25  0.00


In [14]:
colors = kairo.get_dbscan_state_colors(dbscan)
frequencies = pd.DataFrame({
    "before": kairo.get_dbscan_state_frequencies(states, end_date=AMOUNT - pd.Timedelta("1s")),
    "after": kairo.get_dbscan_state_frequencies(states, start_date=AMOUNT),
}).fillna(0).astype(int)
print(frequencies.to_string())
print((100 * frequencies / frequencies.sum()).round(1).to_string())
kairo.plot_dbscan_state_frequencies(frequencies["after"], colors).show()

       before  after
state               
0         525      0
1           0    148
noise      23     35
       before  after
state               
0        95.8    0.0
1         0.0   80.9
noise     4.2   19.1


In [15]:
amounts = features.loc[features["active_cases"] > 0, ["time:timestamp", "attr_means:case:amount"]]
amounts.groupby(np.where(amounts["time:timestamp"] >= AMOUNT, "after", "before"))["attr_means:case:amount"].std().round(1)

after     169.4
before     83.7
Name: attr_means:case:amount, dtype: float64

In [16]:
trajectory = kairo.get_dbscan_log_trajectory(states)
print(len(trajectory), "visits")
print(trajectory.groupby("state").agg(visits=("state", "size"), feature_windows=("feature_windows", "sum"), longest=("duration", "max")).to_string())
kairo.plot_dbscan_log_trajectory(trajectory, colors).show()

98 visits
       visits  feature_windows          longest
state                                          
0          23              525 45 days 00:00:00
1          26              148  9 days 12:00:00
noise      49               58  2 days 00:00:00


In [17]:
from sklearn.neighbors import NearestNeighbors
from thesis_figures import write_k_distance, write_log_trajectory, write_state_timeline

values = scaled.drop(columns="time:timestamp").to_numpy()
k_distances = NearestNeighbors(n_neighbors=min_samples).fit(values).kneighbors(values)[0][:, -1]
print("largest k-distances", np.round(np.sort(k_distances)[::-1][:5], 2))
write_k_distance("k_distance_inter", k_distances, eps=eps, min_samples=min_samples)

names = {"noise": "black!35"}
names.update({str(cluster): f"state{'AB'[number - 1]}" for cluster, number in zip(states["cluster"], states["state"]) if cluster != -1})
write_log_trajectory("log_trajectory_inter", trajectory, names, drift=AMOUNT)
write_state_timeline("states_inter", states["time:timestamp"], states["attr_means:case:amount"], states["state"],
                     y_range=(-2.5, 2.5), y_ticks=[-2, 0, 2], y_label="Mean amount", drift=AMOUNT)

largest k-distances [13.06  3.12  2.62  2.39  2.13]


['../../thesis/figures/tikz/states_inter.tex',
 '../../thesis-de/figures/tikz/states_inter.tex']

In [18]:
weekly = kairo.compute_state_distributions(states, signal_window="7D")
weekly_signal = kairo.compute_drift_signal(weekly, divergence="kl", reference="previous")
weekly_scores = weekly_signal.set_index("signal_window")["drift_score"]
windows = states.groupby(states["time:timestamp"].dt.floor("7D")).size()
print(len(weekly), "signal windows, feature windows per signal window: median", windows.median(), "first", windows.iloc[0], "last", windows.iloc[-1])
print(weekly_scores[weekly_scores > 1].round(2).to_string())
weekly.loc["2020-09-17":"2020-10-08"].round(3)

54 signal windows, feature windows per signal window: median 14.0 first 2 last 1
signal_window
2020-01-30 00:00:00+00:00     1.22
2020-02-20 00:00:00+00:00     2.55
2020-03-26 00:00:00+00:00     1.22
2020-04-30 00:00:00+00:00     1.22
2020-05-14 00:00:00+00:00     1.22
2020-07-02 00:00:00+00:00     1.22
2020-07-16 00:00:00+00:00     1.22
2020-08-27 00:00:00+00:00     2.55
2020-09-10 00:00:00+00:00     2.55
2020-09-24 00:00:00+00:00     1.22
2020-10-01 00:00:00+00:00    14.96
2020-12-31 00:00:00+00:00     1.95


state,noise,0,1
signal_window,,,
2020-09-17 00:00:00+00:00,0.000,1.000,0.000
2020-09-24 00:00:00+00:00,0.071,0.929,0.000
2020-10-01 00:00:00+00:00,0.286,0.000,0.714
2020-10-08 00:00:00+00:00,0.214,0.000,0.786


In [19]:
kairo.plot_drift_signal(weekly_signal, divergence="kl", reference="previous").show()

In [20]:
daily_scores = kairo.compute_drift_signal(kairo.compute_state_distributions(states, signal_window="1D"), divergence="kl", reference="previous").set_index("signal_window")["drift_score"]
print("above 1:", (daily_scores > 1).sum(), "above 10:", (daily_scores > 10).sum(), "first above 10:", daily_scores[daily_scores > 10].index.min().date())

above 1: 46 above 10: 10 first above 10: 2020-10-01


In [21]:
monthly = kairo.compute_state_distributions(states, signal_window="30D")
monthly_signal = kairo.compute_drift_signal(monthly, divergence="kl", reference="baseline")
windows = states.groupby(states["time:timestamp"].dt.floor("30D")).size().rename("feature windows")
print("baseline", monthly.mean().round(3).to_dict())
pd.concat([monthly.round(3), windows, monthly_signal.set_index("signal_window")["drift_score"].round(3)], axis=1)

baseline {'noise': 0.079, '0': 0.726, '1': 0.195}


,noise,0,1,feature windows,drift_score
2019-12-10 00:00:00+00:00,0.062,0.938,0.000,16,0.225
2020-01-09 00:00:00+00:00,0.017,0.983,0.000,60,0.272
2020-02-08 00:00:00+00:00,0.067,0.933,0.000,60,0.223
2020-03-09 00:00:00+00:00,0.033,0.967,0.000,60,0.248
2020-04-08 00:00:00+00:00,0.067,0.933,0.000,60,0.223
2020-05-08 00:00:00+00:00,0.033,0.967,0.000,60,0.248
2020-06-07 00:00:00+00:00,0.017,0.983,0.000,60,0.272
2020-07-07 00:00:00+00:00,0.050,0.950,0.000,60,0.233
2020-08-06 00:00:00+00:00,0.033,0.967,0.000,60,0.248
2020-09-05 00:00:00+00:00,0.100,0.817,0.083,60,0.049


In [22]:
kairo.plot_drift_signal(monthly_signal, divergence="kl", reference="baseline").show()

In [23]:
from thesis_figures import write_drift_signal

write_drift_signal("drift_signal_inter_7d", weekly_signal, "7D", AMOUNT, y_max=16, y_ticks=[0, 5, 10, 15])
write_drift_signal("drift_signal_inter_30d", monthly_signal, "30D", AMOUNT, y_max=1.5, y_ticks=[0, 0.5, 1, 1.5])

['../../thesis/figures/tikz/drift_signal_inter_30d.tex',
 '../../thesis-de/figures/tikz/drift_signal_inter_30d.tex']

In [24]:
abstraction = kairo.abstract_drift_signal(weekly_signal, divergence="kl", reference="previous")
print(abstraction)
print(len(abstraction), "characters")

Drift signal: for every signal window, the KL divergence between its state distribution and the previous signal window, the drift score of the signal window. 54 signal windows from 2019-12-26 00:00:00+00:00 to 2020-12-31 00:00:00+00:00.
Drift scores: median 0.074, mean 0.678, highest 14.958.
The signal windows with the highest drift scores:
- signal window starting 2020-10-01 00:00:00+00:00: 14.958
- signal window starting 2020-02-20 00:00:00+00:00: 2.550
- signal window starting 2020-08-27 00:00:00+00:00: 2.550
- signal window starting 2020-09-10 00:00:00+00:00: 2.550
- signal window starting 2020-12-31 00:00:00+00:00: 1.946
633 characters


In [25]:
from thesis_figures import write_abstraction

write_abstraction("abstraction_drift_signal_inter", abstraction)

['../../thesis/figures/tikz/abstraction_drift_signal_inter.tex',
 '../../thesis-de/figures/tikz/abstraction_drift_signal_inter.tex']